In [ ]:
import os
import sys

sys.path.append("../")

envkey = "OMP_NUM_THREADS"
# Set this environment variable to the number of available cores in your machine,
# to get a fast execution of the Einstein Boltzmann Solver
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))
os.environ[envkey] = str(12)
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
from copy import copy, deepcopy
import numpy as np
import seaborn
from getdist.gaussian_mixtures import GaussianND
from getdist import plots
from scipy.optimize import curve_fit
from scipy.interpolate import UnivariateSpline, RectBivariateSpline
from matplotlib import cm, colors
from matplotlib.ticker import LogLocator, FuncFormatter
import niceplots.utils as nicepl


nicepl.initPlot()
Cs = seaborn.color_palette("colorblind")
Cp = seaborn.color_palette("Paired")
Cs

In [ ]:
response8 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_371900_Response_NSUB8.npz")
dTb = response8["deltab"]

k, Pk  = response8["k"], response8["Pkmean"]

#sanitize
mask = ~np.any(np.isnan(Pk), axis=0)
k = k[mask]
Pk = Pk[:, mask]

norm = colors.Normalize(vmin=np.min(dTb), vmax=np.max(dTb))
cmap = cm.cividis
sm = cm.ScalarMappable(norm=norm, cmap=cmap)

ax = plt.subplot()

Ts = iter(dTb)
for Pki in Pk:
    plt.loglog(k, Pki, c=cmap(norm(next(Ts))))

plt.xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P_\mathrm{TT}(k)\,[\mu \mathrm{K}^2\,h^{-3}\,\mathrm{Mpc}^{3}]$")
plt.title("[CII], $z=1$")
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\delta T_\mathrm{b}\,[\mu\mathrm{K}]$")


In [ ]:
def linear(x, a, b):
    return a * x + b

pobs, pcov = [], []
for Pik in Pk.T:
    pi, ci = curve_fit(linear, dTb, Pik)
    pobs.append(pi)
    pcov.append(ci)
pobs = np.array(pobs)
pcov = np.array(pcov)

ai, bi = pobs[:, 0], pobs[:, 1]
ri_sims =  0.22836 * ai / bi
si = ri_sims * np.sqrt(pcov[:, 0, 0]/ ai**2 + pcov[:, 1, 1]/ bi**2 - 2 * bi / ai * pcov[:, 0, 1])
k_sims = k

plt.errorbar(k, ri_sims, si, ls="--")
plt.xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{dlog}P_\mathrm{TT}(k)/\mathrm{d}\delta_\mathrm{b}$")
plt.semilogx()

In [ ]:
nsub1 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_349406_JK.npz")
nsub2 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_348253_NSUB2_JK.npz")
nsub3 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_348254_NSUB3_JK.npz")
nsub4 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_346009_NSUB4_JK.npz")

color = iter(Cs)
for i, f in zip([1, 2, 3, 4], [nsub1, nsub2, nsub3, nsub4]):
    kc = np.sqrt(f["kedges"][1:] * f["kedges"][:-1])

    cov_i = f["Cov"]
    if np.ndim(cov_i) == 3:
        cov_i = cov_i[0, :, :]

    covclean = cov_i[~np.isnan(cov_i)]
    covclean = covclean.reshape(int(np.sqrt(len(covclean))), int(np.sqrt(len(covclean))))
    kclean = kc[~np.isnan(cov_i[:, 1])]
    plt.loglog(kclean, np.diag(covclean) / i**3, label=r"$N_\mathrm{{part}}={}$".format(i), c=next(color))

plt.xlabel(r"$k_1\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{Cov}(k_1, k_1)\,[\mu \mathrm{K}^4\,h^{-6}\,\mathrm{Mpc}^{6}]$")
plt.title("[CII], $z=1$")
plt.legend()
plt.tight_layout()

In [ ]:
smallboxcovs = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_346009_NSUB4_JK.npz")

cov = np.mean(smallboxcovs["Cov"], axis=0)

valid = ~np.all(np.isnan(cov), axis=1)
cov_smallbox = cov[valid][:, valid]

corr_smallbox = cov_smallbox / np.sqrt(
    np.outer(np.diag(cov_smallbox), np.diag(cov_smallbox))
)

kedges = smallboxcovs["kedges"]
k_smallbox = np.sqrt(kedges[1:] * kedges[:-1])[valid]
kedges_small = kedges[np.r_[valid, False]]

corr_masked = np.ma.masked_invalid(corr_smallbox)

pc = plt.pcolormesh(kedges_small, kedges_small, corr_masked,
                    shading="auto", cmap="viridis")
plt.colorbar(pc, label='Correlation')
plt.loglog()
plt.xlabel("$k_1\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel("$k_2\,[h\,\mathrm{Mpc}^{-1}]$")
plt.title("[CII], $z=1$")
plt.show()

In [ ]:
response4 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_347154_CII_A_Response_NSUB4.npz")

k_subbox, Pk  = response4["k"][1:], response4["Pkmean"][:, 1:]
Pkmean = Pk.mean(axis=0)
dPk = Pk - Pkmean
cov_subbox = 1 / (Pk.shape[0]-1) * np.sum(dPk[:, :, None] * dPk[:, None, :], axis=0)

mask = ~np.all(np.isnan(cov_subbox), axis=0)
k_subbox = k_subbox[mask]
cov_subbox = cov_subbox[np.outer(mask, mask)].reshape(17, 17)

plt.loglog(k_subbox, np.diag(cov_subbox))
plt.loglog(k_smallbox, np.diag(cov_smallbox))

In [ ]:
i = -1
print(k_subbox[i], k_smallbox[i])
plt.loglog(k_subbox, cov_subbox[i, :])
plt.loglog(k_smallbox, cov_smallbox[i, :])

In [ ]:
revenge8fold = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_183794_CO21_Response_NSUB8.npz")
k, Pk = revenge8fold["k"], revenge8fold["Pkmean"]
def get_cov():
    mPk = np.mean(Pk, axis=0)
    dPk = Pk - mPk
    return 1/(512 - 1) * np.sum(dPk[:, :, None] * dPk[:, None, :], axis=0)
k, cov8 = k[0, :], get_cov()

In [ ]:
response4 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_347154_CII_A_Response_NSUB4.npz")

k_subbox, Pk  = response4["k"][1:], response4["Pkmean"][:, 1:]
Pkmean = Pk.mean(axis=0)
dPk = Pk - Pkmean
cov_subbox = 1 / (Pk.shape[0]-1) * np.sum(dPk[:, :, None] * dPk[:, None, :], axis=0)

mask = ~np.all(np.isnan(cov_subbox), axis=0)
k_subbox = k_subbox[mask]
cov_subbox = cov_subbox[np.outer(mask, mask)].reshape(17, 17)

plt.loglog(k_subbox, np.diag(cov_subbox))
plt.loglog(k_smallbox, np.diag(cov_smallbox))
plt.loglog(k, np.diag(cov8) / 4**3)

In [ ]:
# Import Main modules. This might take some time as some functions compile before time
from SSLimPy.interface import sslimpy
from SSLimPy.cosmology import cosmology
from SSLimPy.cosmology import halo_model
from SSLimPy.cosmology import astro
from SSLimPy.LIMsurvey import covariance as scov
from SSLimPy.LIMsurvey import power_spectrum as spobs

In [ ]:
settings = {
    "code":"class", # The Einstein--Boltzman solver that should be used
    "do_RSD" : False, # If RSD should be considerd
    "nonlinearRSD" : False, # If you want to add FOG to the RSD
    "QNLpowerspectrum": False, # Use dewiggled power spectrum (vlasov approximation of nonlinear structure formation)
    "FoG_damp" : "ISTF_like", # The particular parametrization for the FOG. Check PowerSpectrum for the full list
    "halo_model_PS" : True, # If the cosmological shotnoise should be computed from the halo model 
    "output" : ["Power spectrum", "Covariance"], # What output one wants (here power spectrum and Gaussian covariance only)
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 50 * u.Mpc**-1,
    "nk": 200,
}

cosmodict={
    "h": 0.677,
    "Omegam": 0.309167,
    "Omegab": 0.04903,
    "sigma8":0.8222,
    "ns":0.96824,
    "mnu":0.06,
    "Neff":3.044,
}

# Parameters that enter your halo model. Typically they are not changed but you could
halodict={
    "halo_tracer" : "clustering", # Computes all halo quantities from the matter field - neutrinos
    "hmf_model": "ST", # Sheth--Tormann halo mass function
    "concentration": "Diemer19", # Diemer19 halo concentration relation
    "bias_model": "ST99",
}

# Parameters for the Survey specifications
z = np.array([1])
def get_specs(nu):
        nuObs = nu / (z + 1)
        DeltaDeltanu = 0.05143

        surveyspecs = {
                "Tsys_NEFD": 0 * u.uK, #System temperature for instrumental shotnoise
                "Nfeeds": 19,
                "tobs": 1300 * u.h,
                "nD": 1, # Observational parameters
                "beam_FWHM": 0. * u.arcmin,
                "nu":  nu,
                "nuObs": nuObs, # Observed Frequency
                "Delta_nu": DeltaDeltanu * nuObs, # Frequency Bin
                "dnu": 0. * u.MHz, # Spectrograph resolution
                "Omega_field": 18.64 * u.deg**2, # Angular size of survey
        }
        return surveyspecs

astrodict_CII={
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {
        "a": 0.8475,
        "b": 7.2203,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37,
    "meanperserve_scatter": False,
}

nu = 1.897 * u.THz
surveyspecs_CII = get_specs(nu)


In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
    astropars=astrodict_CII,
    obspars_dict=surveyspecs_CII,
)

myastro = myssl.current_astro
mycosmo = myastro.cosmology
h = mycosmo.h()

In [ ]:
pobs = spobs.PowerSpectra(myastro)
ssc_cov = scov.SuperSampleCovariance(pobs)

In [ ]:
kpk = np.loadtxt("/home/sefa/Desktop/LIM-Code/clusterdata/342461_powerspectrum.txt")
k = kpk[:, 0]
Pk = kpk[:, 1]

kth = k * myastro.Mpch**-1
I11 = myastro.Thalo(1, kth, p=1, scale=(1,), beta=1)
I02 = myastro.Thalo(1, kth, p=1, scale=(2,), beta=0)
plin = mycosmo.matpow(kth, 1)

color = iter(Cs)
plt.loglog(k, Pk, c=next(color), label="Simulation")
plt.loglog(k, (I11**2 * plin).to(myastro.Mpch**3 * u.uK**2), c=next(color), label=r"$P_\mathrm{line}^\mathrm{tree}$")
plt.loglog(k, (I02).to(myastro.Mpch**3 * u.uK**2), c=next(color), label=r"$P_\mathrm{line}^\mathrm{1h}$")
plt.loglog(k, (I11**2 * plin + I02).to(myastro.Mpch**3 * u.uK**2), c=next(color), label=r"$P_\mathrm{line}$")

plt.legend()
plt.xlabel("$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel("$P(k)\,[\mu \mathrm{K}^2 \,h^{-3}\,\mathrm{Mpc}^{3}]$")

In [ ]:
def get_ssl_response(kr):
    Delta = 4 * np.pi / (2 * np.pi)**3 * mycosmo.k**3 * mycosmo.matpow(mycosmo.k, 1, myastro.halomodel.tracer)
    logD = np.log(Delta.to(1).value)

    gamma = UnivariateSpline(np.log(mycosmo.k.value), logD, s=0).derivative(1)(np.log(kr.to(mycosmo.k.unit).value))
    Pk = myastro.cosmology.matpow(kr, 1, myastro.halomodel.tracer)

    I11 = myastro.Thalo(1, kr, p=1, scale=(1,), beta=1)
    I21 = (
        myastro.Thalo(1, kr, p=1, scale=(1,), beta="b2")
        + 4 / 3 * myastro.Thalo(1, kr, p=1, scale=(1,), beta="bG2")
    )
    
    I02 = myastro.Thalo(1, kr, p=1, scale=(2,), beta=0)
    I12 = myastro.Thalo(1, kr, p=1, scale=(2,), beta=1)

    Phalo = I11**2 * Pk# + I02

    BC = (68 / 21 * I11**2 + 2 * I11 * I21) * Pk
    LD = - 1 / 3 * gamma * I11**2 * Pk
    HSV = 0 #I12
    return (BC + LD + HSV) / Phalo, Phalo

In [ ]:
from scipy.special import spherical_jn
from scipy.integrate import dblquad

def A_of_u(u, epsabs=1e-6, epsrel=1e-6):
    """
    Compute A(u) = ⟨ j0^2(u kx/2) j0^2(u ky/2) j0^2(u kz/2) ⟩ over the sphere
    using scipy.integrate.dblquad.
    """

    # integrand in φ then θ (dblquad integrates inner first)
    def integrand(phi, theta):
        # direction cosines
        kx = np.sin(theta)*np.cos(phi)
        ky = np.sin(theta)*np.sin(phi)
        kz = np.cos(theta)

        Jx = spherical_jn(0, 0.5*u*kx)
        Jy = spherical_jn(0, 0.5*u*ky)
        Jz = spherical_jn(0, 0.5*u*kz)

        return (Jx*Jy*Jz)**2 * np.sin(theta)   # include Jacobian factor

    # dblquad integrates phi from 0→2π inside, θ from 0→π outside
    res, err = dblquad(integrand, 0, np.pi, lambda t: 0, lambda t: 2*np.pi,
                       epsabs=epsabs, epsrel=epsrel)

    return res / (4*np.pi)

In [ ]:
L = ssc_cov.survey_specs.Lfield()

x = np.geomspace(1e-2, 1e2)
k = x / L
A = [A_of_u(xi) for xi in x]

sigma_integrand = 4 * np.pi * k**3 * mycosmo.matpow(k, 1) * A
soverV = np.trapz(sigma_integrand, x=np.log(k.value)) / (2 * np.pi)**3

In [ ]:
kr = (np.geomspace(np.min(k_subbox), np.max(k_subbox)) * myastro.Mpch**-1).to(myastro.Mpch**-1)

response_ssl, P_halo = get_ssl_response(k_smallbox * myastro.Mpch**-1)
ssl_ssc = soverV * np.outer(response_ssl * P_halo, response_ssl * P_halo)
ssl_ssc = ssl_ssc.to(myastro.Mpch**6 * u.uK**4)

In [ ]:
nomean = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_366344_CII_A_noMean_Response_NSUB2.npz")

In [ ]:
nomean.keys()

In [ ]:
response8 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_189373_CII_A_Powerspectrum.npz")
k, Pk  = response8["k"], response8["Pk"]
plt.loglog(k, Pk, label="Simulation")
plt.loglog(nomean["k"], nomean["Pkmean"][0, :], "k--", label="no mean substraction")
plt.loglog(k_smallbox, P_halo.to(myastro.Mpch**3 * u.uK**2) / myastro.Tavg(1)**2 * 0.22836**2, label=r"$P_{2\mathrm{h}}$")
plt.loglog(k_smallbox, myastro.Thalo(1, k_smallbox * myastro.Mpch**-1, p=1, scale=(2,)), label=r"$P_{SN}$")
plt.legend()

In [ ]:
response4 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/run_347154_CII_A_Response_NSUB4.npz")

k_subbox, Pk  = response4["k"][1:], response4["Pkmean"][:, 1:]
Pkmean = Pk.mean(axis=0)
dPk = Pk - Pkmean
cov_subbox = 1 / (Pk.shape[0]-1) * np.sum(dPk[:, :, None] * dPk[:, None, :], axis=0)

mask = ~np.all(np.isnan(cov_subbox), axis=0)
k_subbox = k_subbox[mask]
cov_subbox = cov_subbox[np.outer(mask, mask)].reshape(17, 17)

plt.loglog(k_subbox, np.diag(cov_subbox))
plt.loglog(k_smallbox, np.diag(cov_smallbox))
plt.loglog(k_smallbox, np.diag(cov_smallbox + 1 * ssl_ssc.value))

In [ ]:
plt.loglog(k_subbox, cov_subbox[:, 15])
plt.loglog(k_smallbox, (cov_smallbox)[:, 15])
plt.loglog(k_smallbox, (cov_smallbox + ssl_ssc.value)[:, 15])

In [ ]:
f2 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_371898_Response_NSUB2.npz")
f4 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_371899_Response_NSUB4.npz")
f8 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_371900_Response_NSUB8.npz")


In [ ]:
x = np.geomspace(1e-2, 1e2)
A = np.array([A_of_u(xi) for xi in x])

soverV = []
for L in 1024 * myastro.Mpch / np.array([2, 4, 8]):
    k = x / L
    sigma_integrand = 4 * np.pi * k**3 * mycosmo.matpow(k, 1) * A
    soverV.append(np.trapz(sigma_integrand, x=np.log(k.value)) / (2 * np.pi)**3)
soverV = np.array(soverV)

In [ ]:
color = iter(Cs)
for f, Nsub in zip([f2, f4, f8], [2, 4, 8]):
    k, Pk = f["k"], f["Pkmean"]
    nanmask = np.any(np.isnan(Pk), axis=0)
    k = k[~nanmask]
    Pk = Pk[:, ~nanmask]
    lims = np.percentile(Pk, [16, 84], axis=0)
    
    c = next(color)
    plt.loglog(k, np.mean(Pk, axis=0), c=c, label=r"$N_\mathrm{{sub}}={}$".format(Nsub))
    plt.fill_between(k, lims[0, :], lims[1, :], color=c, alpha=0.1)
plt.legend()

plt.xlabel("$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel("$P_\mathrm{TT}\,[\mu \mathrm{K}^2\,h^{-3}\,\mathrm{Mpc}^{3}]$")

In [ ]:
jk8 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_372476_NSUB8_JK.npz")
cov_smallbox8 = np.mean(jk8["Cov"], axis=0)

mask = ~np.all(np.isnan(cov_smallbox8), axis=0)

k8 = np.sqrt(jk8["kedges"][1:] *  jk8["kedges"][:-1])[mask]
cov_smallbox8 = cov_smallbox8[np.outer(mask, mask)].reshape((16, 16))

In [ ]:
color = iter(Cs)
f, Nsub = f8, 8
Pk = f["Pkmean"]
nanmask = np.any(np.isnan(Pk), axis=0)
Pk = Pk[:, ~nanmask]
k, modes = f["k"][~nanmask], f["modes"][0, ~nanmask]
mPk = np.mean(Pk, axis=0)
dPk = Pk - mPk[None, :]
Gcov_esque = 1 / modes * np.outer(mPk, mPk)
cov_normalised = 1/(Nsub**3- 1) * np.einsum("il, im -> lm", dPk, dPk) / Gcov_esque

fig, axs = plt.subplots(2, 1, sharex=True, gridspec_kw={'height_ratios': [5, 2]})

c = next(color)
axs[0].loglog(k, np.diag(cov_normalised) -1, c=c, label=r"Subbox $N_\mathrm{{sub}}={}$".format(Nsub))
inter = UnivariateSpline(np.log(k), np.log(np.diag(cov_normalised) -1), s=0)

c = next(color)
axs[0].loglog(k8, np.diag(cov_smallbox8/ Gcov_esque) -1, c=c, ls="--", label=r"Smallbox $N_\mathrm{sub}=8$")
axs[1].semilogx(k8, 100 * ((np.diag(cov_smallbox8/ Gcov_esque) -1) / np.exp(inter(np.log(k8))))-100 , c=c, ls="--")

c = next(color)
axs[0].loglog(k8, np.diag(cov_smallbox8/ Gcov_esque) + modes * soverV[-1] * ri_sims**2 - 1, c = c, label=r"Smallbox + SSC (Sims)")
axs[1].semilogx(k8, 100 * ((np.diag(cov_smallbox8/ Gcov_esque) + modes * soverV[-1] * ri_sims**2 - 1) / np.exp(inter(np.log(k8)))) - 100, c=c)

c = next(color)
ri, pi = get_ssl_response(k8 * myastro.Mpch**-1)
axs[0].loglog(k8, np.diag(cov_smallbox8/ Gcov_esque) + modes * soverV[-1] * ri**2 - 1, c = c, label=r"Smallbox + SSC (HM)")
axs[1].semilogx(k8, 100* ((np.diag(cov_smallbox8/ Gcov_esque) + modes * soverV[-1] * ri**2 - 1) / np.exp(inter(np.log(k8)))) - 100, c=c)

axs[0].legend()

axs[1].set_xlabel("$k\,[h\,\mathrm{Mpc}^{-1}]$")
axs[0].set_ylabel("$\mathrm{Cov}(k_1,k_1)/ \mathrm{Cov}_\mathrm{G}-1$")
axs[1].set_ylabel("$\%$ Deviation")
plt.tight_layout()
fig.subplots_adjust(hspace=0.0)
# plt.xlim(4e-2, 0.3)
# plt.ylim(1e-2, 10)

In [ ]:
jk8 = np.load("/home/sefa/Desktop/LIM-Code/clusterdata/CII_A_372476_NSUB8_JK.npz")
mask = ~np.any(np.isnan(jk8["Cov"]), axis=0)
covs = np.array([jk8["Cov"][i, :, :][mask].reshape((16, 16)) for i in range(8**3)])
k = np.sqrt(jk8["kedges"][1:] * jk8["kedges"][:-1])[np.any(mask, axis=0)]
dTb = jk8["deltab"]

In [ ]:
norm = colors.Normalize(vmin=np.min(dTb), vmax=np.max(dTb))
cmap = cm.cividis
sm = cm.ScalarMappable(norm=norm, cmap=cmap)

ax = plt.subplot()

Ts = iter(dTb)
for cov in covs:
    plt.loglog(k, np.diag(cov), c=cmap(norm(next(Ts))))

plt.xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{Cov}(k,k)\,[\mu \mathrm{K}^4\,h^{-6}\,\mathrm{Mpc}^{6}]$")
plt.title("[CII], $z=1$")
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(r"$\delta T_\mathrm{b}\,[\mu\mathrm{K}]$")

In [ ]:
plt.scatter(dTb, covs[:, 9, 9], c=cmap(norm(dTb)))
plt.xlabel("$\delta T_\mathrm{b}\,[\mu\mathrm{K}]$")
plt.ylabel("$\mathrm{Cov}_{ij}\,[\mu \mathrm{K}^4\,h^{-6}\,\mathrm{Mpc}^{6}]$")

In [ ]:
def get_ssl_response_both_models(kr):
    Delta = 4 * np.pi / (2 * np.pi)**3 * mycosmo.k**3 * mycosmo.matpow(mycosmo.k, 1, myastro.halomodel.tracer)
    logD = np.log(Delta.to(1).value)

    gamma = UnivariateSpline(np.log(mycosmo.k.value), logD, s=0).derivative(1)(np.log(kr.to(mycosmo.k.unit).value))
    Pk = myastro.cosmology.matpow(kr, 1, myastro.halomodel.tracer)

    I11 = myastro.Thalo(1, kr, p=1, scale=(1,), beta=1)
    I21 = (
        myastro.Thalo(1, kr, p=1, scale=(1,), beta="b2")
        + 4 / 3 * myastro.Thalo(1, kr, p=1, scale=(1,), beta="bG2")
    )
    
    I02 = myastro.Thalo(1, kr, p=1, scale=(2,), beta=0)
    I12 = myastro.Thalo(1, kr, p=1, scale=(2,), beta=1)

    Pclust = I11**2 * Pk
    Phalo = Pclust + I02

    BC = (68 / 21 * I11**2 + 2 * I11 * I21) * Pk
    LD = - 1 / 3 * gamma * I11**2 * Pk
    HSV = I12
    return (BC + LD + HSV) / Phalo, (BC + LD) / Pclust

In [ ]:
plt.errorbar(k, ri_sims, si, ls="--", label="Sims")

kr = np.geomspace(np.min(k), np.max(k))
r1, r2 = get_ssl_response_both_models(kr * myastro.Mpch**-1)
plt.plot(kr, r1, label="Full HM")
plt.plot(kr, r2, label="Only Clustering")
plt.xlabel(r"$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$\mathrm{dlog}P_\mathrm{TT}(k)/\mathrm{d}\delta_\mathrm{b}$")
plt.legend()
plt.semilogx()

In [ ]:
color = iter(Cs)
r, p = get_ssl_response(f2["k"] * myastro.Mpch**-1)
rf, pf = get_ssl_response(0.35 * myastro.Mpch**-1)
for f, Nsub, sV in zip([f2, f4, f8], [2, 4, 8], soverV):
    k, Pk = f["k"], f["Pkmean"]
    nanmask = np.any(np.isnan(Pk), axis=0)
    k = k[~nanmask]
    Pk = Pk[:, ~nanmask]
    
    mPk = np.mean(Pk, axis=0)
    dPk = Pk - mPk[None, :]

    cov = 1/(Nsub**3- 1) * np.einsum("il, im -> lm", dPk, dPk) 
    inter = RectBivariateSpline(k, k, cov, s=0) 

    c = next(color)
    plt.loglog(f2["k"], r * p * rf * pf * sV, c=c, ls="--")
    plt.loglog(k, inter(0.35, k, grid=False), c=c, label=r"$N_\mathrm{{sub}}={}$".format(Nsub))
plt.legend()


plt.xlabel("$k\,[h\,\mathrm{Mpc}^{-1}]$")
plt.ylabel("$\mathrm{Cov}(k_1,k_1)\,[\mu \mathrm{K}^4\,h^{-6}\,\mathrm{Mpc}^{6}]$")